# DS2002 · Walmart Warm-Up SKU Mess

**Lab — 2026-10-16 · Fall 2026**  

---

## Lab 08 — Walmart Warm-Up (SKU Mess)

This lab uses the **real midterm data** in `../06-midterm-walmart/data/`. It's a warm-up so you hit the ground running on the project. On Kaggle/Colab, upload the data folder or adjust the path.

*If the file isn't found, the cell falls back to a small inline sample so the notebook still runs.*

In [4]:
import pandas as pd
PATH = '/content/product_catalog.csv'
try:
   cat = pd.read_csv(PATH)
except FileNotFoundError:
   from io import StringIO
   cat = pd.read_csv(StringIO('sku,product_name,category\nPT-12,pop-tarts strawberry,Breakfast Foods\nSKU#459812,Strawberry Pop-Tart,Breakfast\nPOPTART-STRAW,Pop-Tarts Strawberry,Breakfast'))
cat.head()


,sku,product_name,category,subcategory,unit_cost,supplier,package_size
0,PT-12,pop-tarts strawberry,Breakfast Foods,NaN,1.93,Johnson & Johnson,4-pack
1,SKU#459812,Strawberry Pop-Tart,Breakfast,Pop-Tarts,2.26,Roku Inc,4pk
2,POPTART-STRAW,Pop-Tarts Strawberry,Breakfast,NaN,2.11,Sara Lee,6 pack
3,PT_STRAWBERRY,pop-tarts strawberry,breakfast,Pop-Tarts,NaN,Campbell's,4 pack
4,WTR-24,BOTTLED WATER 24PK,Emergency Supplies,Bottled,2.64,Sara Lee,1 pack


### Q1 — How many rows look like Pop-Tart products?

Filter on `product_name` containing `pop`, case-insensitively. Print the count and the matching rows.

In [5]:
pop_rows = cat[
    cat['product_name'].str.lower().str.contains('pop', na=False)
]

print("Number of Pop-Tart products:", len(pop_rows))
print(pop_rows)

Number of Pop-Tart products: 4
             sku          product_name         category subcategory  \
0          PT-12  pop-tarts strawberry  Breakfast Foods         NaN   
1     SKU#459812   Strawberry Pop-Tart        Breakfast   Pop-Tarts   
2  POPTART-STRAW  Pop-Tarts Strawberry        Breakfast         NaN   
3  PT_STRAWBERRY  pop-tarts strawberry        breakfast   Pop-Tarts   

   unit_cost           supplier package_size  
0       1.93  Johnson & Johnson       4-pack  
1       2.26           Roku Inc          4pk  
2       2.11           Sara Lee       6 pack  
3        NaN         Campbell's       4 pack  


### Q2 — Which SKUs and names all mean *strawberry* Pop-Tart?

List them. Then note which Pop-Tart rows are **not** strawberry — those must stay separate, and saying so is part of the answer.

In [6]:
strawberry_rows = pop_rows[
    pop_rows['product_name'].str.lower().str.contains('strawberry', na=False)
]

print("Strawberry Pop-Tart variants:")
print(strawberry_rows[['sku', 'product_name']])

print("\nPop-Tart rows that are a different product:")
print(
    pop_rows[
        ~pop_rows['product_name'].str.lower().str.contains('strawberry', na=False)
    ][['sku', 'product_name']]
)

Strawberry Pop-Tart variants:
             sku          product_name
0          PT-12  pop-tarts strawberry
1     SKU#459812   Strawberry Pop-Tart
2  POPTART-STRAW  Pop-Tarts Strawberry
3  PT_STRAWBERRY  pop-tarts strawberry

Pop-Tart rows that are a different product:
Empty DataFrame
Columns: [sku, product_name]
Index: []


**Pop-Tart rows that are a different product:** _..._

### Q3 — Build the mapping

A dict from each variant SKU to one canonical SKU. Then apply it in a way that leaves unmapped SKUs as themselves rather than turning them into `NaN` — that single mistake silently deletes most of the catalog.

In [7]:
mapping = {
    'SKU#459812': 'PT-12',
    'POPTART-STRAW': 'PT-12'
}

cat['canonical_sku'] = cat['sku'].replace(mapping)

print(cat[['sku', 'product_name', 'canonical_sku']])

              sku            product_name  canonical_sku
0           PT-12    pop-tarts strawberry          PT-12
1      SKU#459812     Strawberry Pop-Tart          PT-12
2   POPTART-STRAW    Pop-Tarts Strawberry          PT-12
3   PT_STRAWBERRY    pop-tarts strawberry  PT_STRAWBERRY
4          WTR-24      BOTTLED WATER 24PK         WTR-24
5       WATER24PK   Bottled Water 24-Pack      WATER24PK
6        WTR_24PK         Water - 24 Pack       WTR_24PK
7        BATT-100        BATTERIES AA 4PK       BATT-100
8         BAT-AA4     AA Batteries 4-Pack        BAT-AA4
9     BATT_AA_4PK     AA Batteries 4-Pack    BATT_AA_4PK
10       FLASH-01          FLASHLIGHT LED       FLASH-01
11      LED-FLASH          FLASHLIGHT LED      LED-FLASH
12        BEER-24    Light Beer (24-pack)        BEER-24
13       LTBEER24    Light Beer (24-pack)       LTBEER24
14        BG-MONO     Monopoly Board Game        BG-MONO
15    MONOPOLY-01     Monopoly Board Game    MONOPOLY-01
16      GAME-MONO   Board Game 

### Q4 — Normalize `category`

Strip and title-case, then show the distinct values before and after with counts. Note any pair that is still two labels for one thing — `Breakfast` and `Breakfast Foods` are not the same string but they are the same category.

In [8]:
print("Categories before normalization:")
print(cat['category'].value_counts(dropna=False))

cat['category'] = cat['category'].str.strip().str.title()

print("\nCategories after normalization:")
print(cat['category'].value_counts(dropna=False))

print("\nDistinct categories after normalization:")
print(sorted(cat['category'].dropna().unique()))

Categories before normalization:
category
Emergency              5
Emergency Supplies     3
EMERGENCY              3
ENTERTAINMENT          3
GROCERY                3
Breakfast              2
entertainment          2
household              2
ELECTRONICS            2
health                 2
Baby                   2
Electronics            1
ALCOHOL                1
Alcoholic Beverages    1
breakfast              1
Breakfast Foods        1
Groceries              1
Grocery                1
electronics            1
snacks                 1
grocery                1
HOUSEHOLD              1
Snacks                 1
Household              1
baby                   1
Baby Products          1
emergency              1
Name: count, dtype: int64

Categories after normalization:
category
Emergency              9
Entertainment          5
Grocery                5
Electronics            4
Household              4
Emergency Supplies     3
Breakfast              3
Baby                   3
Snacks         

### Q5 — Find the fragmentation you have not spotted yet

Hand-scanning a catalog does not scale. **TODO:** build a candidate list programmatically — normalize each product name (lowercase, strip punctuation, sort the words) and group by that, then show any group holding more than one SKU.

This is the workflow from Monday's lecture, and it is how you will actually approach the full catalog on the midterm.

In [9]:
import re

def normalize_name(name):
    name = str(name).lower().strip()
    name = re.sub(r'[^\w\s]', '', name)
    words = name.split()
    return ' '.join(sorted(words))

cat['normalized_name'] = cat['product_name'].apply(normalize_name)

fragmented = (
    cat.groupby('normalized_name')
    .agg(
        sku_count=('sku', 'nunique'),
        skus=('sku', list),
        product_names=('product_name', list)
    )
    .reset_index()
)

fragmented = fragmented[fragmented['sku_count'] > 1]

print(fragmented.to_string(index=False))

    normalized_name  sku_count                                  skus                                                      product_names
  24pack beer light          2                   [BEER-24, LTBEER24]                       [Light Beer (24-pack), Light Beer (24-pack)]
 4pack aa batteries          2                [BAT-AA4, BATT_AA_4PK]                         [AA Batteries 4-Pack, AA Batteries 4-Pack]
board game monopoly          3     [BG-MONO, MONOPOLY-01, GAME-MONO]  [Monopoly Board Game, Monopoly Board Game, Board Game - Monopoly]
        bread white          2                [BREAD-WH, WHITEBREAD]                                         [WHITE BREAD, WHITE BREAD]
      butter peanut          2                   [PB-16, PNUTBTR-16]                                     [PEANUT BUTTER, Peanut Butter]
  candles emergency          2                [CNDL-EM, CANDLE-EMRG]                           [Candles (Emergency), Emergency Candles]
        canned soup          2                 [

### Q6 — Prove the mapping is safe

**TODO:** assert that applying your mapping produced no nulls in the canonical column and did not change the number of rows. Consolidation renames; it never adds or removes.

In [10]:
assert cat['canonical_sku'].notna().all(), "Error: Null values found in canonical_sku"
assert len(cat['canonical_sku']) == len(cat), "Error: Row count changed"

print("Mapping is safe.")
print("Original row count:", len(cat))
print("Canonical column nulls:", cat['canonical_sku'].isna().sum())

Mapping is safe.
Original row count: 45
Canonical column nulls: 0


### Write-up

**a)** Name two product families beyond Pop-Tarts that appear fragmented, and how you would group them.

**b)** Where does consolidation stop being a data-cleaning task and become a business decision? Give one example from this catalog where you genuinely are not sure, and say who you would ask.

A)The board games and the chips variety pack. I would check to see if the board games are all the same and can be grouped together. Same with the variety pack of chips, just seeing if they are all the same category. Then i would grouo based on flavors and types of game.

b) Consolidation is a business decision when the data doesn't distinguish if the products are actually different. I would not know if the streaming stick and the roku streaming stick are different products or just labeled differently.

_your answer here_